In [1]:
import pandas as pd

train = pd.read_parquet("train-00000-of-00001.parquet")
validation = pd.read_parquet("validation-00000-of-00001.parquet")
test = pd.read_parquet("test-00000-of-00001.parquet")

print(train.head())
print(train.shape)
print(train.columns)

FileNotFoundError: train-00000-of-00001.parquet

In [ ]:
print("Training data:", train.shape)
print("Validation data:", validation.shape)
print("Testing data:", test.shape)

In [ ]:
train.isnull().sum()

In [2]:
train["class_label"].value_counts()

NameError: name 'train' is not defined

In [3]:
print("Before cleaning:", train.shape)

print(train.isnull().sum())

NameError: name 'train' is not defined

In [9]:
train = train.dropna(subset=["tweet_text", "class_label"])

validation = validation.dropna(
    subset=["tweet_text", "class_label"]
)

test = test.dropna(
    subset=["tweet_text", "class_label"]
)

print("After cleaning:", train.shape)

After cleaning: (53531, 2)


In [10]:
train = train.drop_duplicates(
    subset=["tweet_text", "class_label"]
)

print("After removing duplicates:", train.shape)

After removing duplicates: (53531, 2)


In [11]:
import re

def clean_text(text):
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove special characters and punctuation
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [12]:
train["clean_text"] = train["tweet_text"].apply(clean_text)

validation["clean_text"] = validation["tweet_text"].apply(clean_text)

test["clean_text"] = test["tweet_text"].apply(clean_text)

In [13]:
train[["tweet_text", "clean_text"]].head(10)

,tweet_text,clean_text
0,Powerful Ecuador quake kills at least 235: POR...,powerful ecuador quake kills at least 235 port...
1,Im at awe and saddened with the #EcuadorEarthq...,im at awe and saddened with the ecuadorearthqu...
2,RT @RachelAndJun: Our hearts are with everyone...,rt rachelandjun our hearts are with everyone i...
3,RT @noticias2000: Ecuador quake death toll has...,rt noticias2000 ecuador quake death toll has r...
4,RT @pzf: BREAKING PHOTOS: Major damage reporte...,rt pzf breaking photos major damage reported i...
5,"Official death toll now 233, says President Co...",official death toll now 233 says president cor...
6,RT @thejpc: Israeli relief org already in Japa...,rt thejpc israeli relief org already in japan ...
7,RT @SamaritansPurse: We are rushing aid to hel...,rt samaritanspurse we are rushing aid to help ...
8,RT @GConnaughton: #Toronto &amp; #Canadian fir...,rt gconnaughton toronto amp canadian first res...
9,"#Ecuador #earthquake in #USA, 272 people killed",ecuador earthquake in usa 272 people killed


In [14]:
print(train[["tweet_text", "clean_text", "class_label"]].head())

print("Missing cleaned text:",
      train["clean_text"].isnull().sum())

                                          tweet_text  \
0  Powerful Ecuador quake kills at least 235: POR...   
1  Im at awe and saddened with the #EcuadorEarthq...   
2  RT @RachelAndJun: Our hearts are with everyone...   
3  RT @noticias2000: Ecuador quake death toll has...   
4  RT @pzf: BREAKING PHOTOS: Major damage reporte...   

                                          clean_text  \
0  powerful ecuador quake kills at least 235 port...   
1  im at awe and saddened with the ecuadorearthqu...   
2  rt rachelandjun our hearts are with everyone i...   
3  rt noticias2000 ecuador quake death toll has r...   
4  rt pzf breaking photos major damage reported i...   

                              class_label  
0                  injured_or_dead_people  
1  rescue_volunteering_or_donation_effort  
2                    sympathy_and_support  
3                  injured_or_dead_people  
4       infrastructure_and_utility_damage  
Missing cleaned text: 0


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2)
)

In [16]:
X_train = tfidf.fit_transform(train["clean_text"])

In [17]:
X_validation = tfidf.transform(validation["clean_text"])

X_test = tfidf.transform(test["clean_text"])

In [18]:
y_train = train["class_label"]

y_validation = validation["class_label"]

y_test = test["class_label"]

In [19]:
print("Training features:", X_train.shape)
print("Validation features:", X_validation.shape)
print("Testing features:", X_test.shape)

print("Number of vocabulary words:", len(tfidf.vocabulary_))

Training features: (53531, 10000)
Validation features: (7793, 10000)
Testing features: (15160, 10000)
Number of vocabulary words: 10000


In [20]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [21]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_validation)

accuracy = accuracy_score(y_validation, y_pred)

print("Validation Accuracy:", accuracy)

print(classification_report(
    y_validation,
    y_pred,
    zero_division=0
))

Validation Accuracy: 0.7225715385602464
                                        precision    recall  f1-score   support

                    caution_and_advice       0.60      0.73      0.66       550
      displaced_people_and_evacuations       0.84      0.86      0.85       409
     infrastructure_and_utility_damage       0.76      0.79      0.78       831
                injured_or_dead_people       0.89      0.89      0.89       746
               missing_or_found_people       0.53      0.78      0.63        36
                      not_humanitarian       0.50      0.62      0.55       644
            other_relevant_information       0.55      0.46      0.50      1236
              requests_or_urgent_needs       0.44      0.70      0.54       264
rescue_volunteering_or_donation_effort       0.87      0.77      0.81      2168
                  sympathy_and_support       0.82      0.78      0.80       909

                              accuracy                           0.72      779

In [22]:
y_test_pred = model.predict(X_test)

print("Test Accuracy:",
      accuracy_score(y_test, y_test_pred))

print(classification_report(
    y_test,
    y_test_pred,
    zero_division=0
))

Test Accuracy: 0.7299472295514512
                                        precision    recall  f1-score   support

                    caution_and_advice       0.62      0.71      0.66      1070
      displaced_people_and_evacuations       0.84      0.89      0.86       790
     infrastructure_and_utility_damage       0.79      0.81      0.80      1617
                injured_or_dead_people       0.89      0.89      0.89      1447
               missing_or_found_people       0.51      0.85      0.64        72
                      not_humanitarian       0.51      0.63      0.57      1245
            other_relevant_information       0.57      0.48      0.52      2407
              requests_or_urgent_needs       0.42      0.72      0.53       521
rescue_volunteering_or_donation_effort       0.87      0.77      0.82      4219
                  sympathy_and_support       0.83      0.77      0.80      1772

                              accuracy                           0.73     15160
    

In [23]:
new_report = [
    "People are trapped inside buildings after a severe earthquake"
]

new_report_tfidf = tfidf.transform(new_report)

prediction = model.predict(new_report_tfidf)

print("Predicted Category:", prediction[0])

Predicted Category: infrastructure_and_utility_damage


In [24]:
probabilities = model.predict_proba(new_report_tfidf)

print("Predicted Category:", model.classes_[probabilities.argmax()])
print("Confidence:", probabilities.max())

Predicted Category: infrastructure_and_utility_damage
Confidence: 0.2905248138061226


In [ ]:
import joblib

joblib.dump(model, "../models/disaster_classifier.pkl")

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

In [26]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

In [27]:
joblib.dump(model, "../models/disaster_classifier.pkl")

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("Both models saved successfully!")

Both models saved successfully!


In [28]:
print(os.path.exists("../models/disaster_classifier.pkl"))
print(os.path.exists("../models/tfidf_vectorizer.pkl"))

True
True
